# MDA-MB-468 — CyTOFSTD ingestion and comparison of three `cytof_transform` normalizations

**Data:** `Data/MDAMB/MDAMB468_Yael_cols_corrected.csv` — 67,563 cells x 32 markers, **raw (linear, pre-arcsinh) intensities**,
extracted from `CyTOF_Data/CyTOF1_MDAMB468_Yael.fcs`.

**Goal:** ingest with `cytofstandard`, apply three normalization variants from `cytof_transform` 0.2.0 to the
*same* QC'd cells, and decide which is best.

| label | call | technical factor | correction |
|---|---|---|---|
| **divide** (B1) | `method="divide"` | `M` = std-minimizing convex-weighted mean of mean-normalized core histones | `x / M` on **raw** counts, **then** `arcsinh(x/5)` |
| **regress single-γ** (A) | `method="regress"`, `gamma_mode="single"` | `f` = PC1 of core histones | `x − γ̄·(f − med f)`, one shared γ̄ for every marker |
| **regress per-marker γ** (A) | `method="regress"`, `gamma_mode="per_marker"` | `f` = PC1 of core histones | `x − γ_m·(f − med f)`, γ_m fitted per marker |

These three are not an arbitrary set — they factor the design along two independent axes:

- **divide vs regress single-γ** — same *uniform* correction applied to every marker, but multiplicative on raw
  counts vs additive in arcsinh space. Isolates the **form and scale** of the correction.
- **regress single-γ vs regress per-marker γ** — same additive arcsinh form, but one shared slope vs a slope
  fitted per marker. Isolates **uniform vs per-marker sensitivity**.

**A fourth arm was tested and rejected — do not re-add it.** `protect_covariates=["KI67","H3S28p"]` fits each
marker on `[f, KI67, H3S28p]` jointly and subtracts only the `f` term, the intent being to spare the
proliferation axis. On MDA-MB-468 it shrank γ for 92% of markers (mean −0.195) and drove several **negative**
(H3K27ac 0.479→−0.021, H2AK119ub 0.155→−0.123), which is physically meaningless: γ<0 means the correction
*adds* signal to cells that stained more. The cause is collinearity — KI67 and H3S28p themselves correlate
0.51 and 0.60 with the technical factor, so conditioning on them collapses the `f` coefficient panel-wide.
The result is systematic under-correction that superficially reads as biology preservation. This is also why
the covariates were a bad choice on their own terms: KI67 and H3S28p encode cell-cycle **biology**, not
permeability.

**Design decisions**
- QC is a single floor: core histones `H3`, `H4`, `H3.3` all `> 5` on raw data. Everything else was handled upstream.
- `control_markers` = the three core histones. `markers_to_correct` = the **25 intracellular** non-core-histone
  markers, identical across methods. The 4 surface markers (`CD24`, `CD44`, `CD49f`, `EpCAM`) are **not**
  corrected — permeabilization affects intracellular staining only — which also makes them a negative control
  (see section 4).
- After normalization the **core histones are dropped** and never used again.
- UMAP + Leiden are built on the **16 epigenetic modifications only**; marker plots are shown for **all** markers.

## 0. Setup

In [ ]:
import sys, shutil, warnings, json
from pathlib import Path

# cytof_transform is not pip-installed; it is used via sys.path injection.
# NOTE: deliberately do NOT add /Code to sys.path. If `CyTOFHelper` becomes importable,
# cytof_transform._cytof_divide_b1 silently swaps in the legacy normalize_data and the
# reported permeabilization factor stops matching the applied correction.
CT_PATH = "/Users/ronguy/Dropbox/Work/CyTOF/Code/cytof-transform"
if CT_PATH not in sys.path:
    sys.path.insert(0, CT_PATH)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as ss

import scanpy as sc

import cytof_transform
from cytofstandard import Project

warnings.filterwarnings("ignore", category=UserWarning, module="umap")
sc.settings.verbosity = 1
%matplotlib inline

plt.rcParams.update({
    'axes.labelsize': 11, 'axes.titlesize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'figure.dpi': 110, 'pdf.fonttype': 42, 'ps.fonttype': 42,
})
sns.set_style("white")

print("cytof_transform", cytof_transform.__version__, "->", cytof_transform.__file__)
try:
    import CyTOFHelper  # noqa: F401
    print("!! WARNING: CyTOFHelper is importable — the divide path will use the LEGACY "
          "implementation, not the native std-min divide documented in this notebook.")
except ModuleNotFoundError:
    print("CyTOFHelper not importable -> native std-min divide will be used (intended).")

In [ ]:
def uns_history(adata, key):
    """Read uns[key]['history'] as a list of dicts.

    AnnData round-trips these records through zarr as JSON *strings*, so a freshly
    written history holds dicts but a reloaded one holds str. Normalize both.
    """
    node = adata.uns.get(key, None)
    if node is None or not hasattr(node, "get"):
        return []
    hist = node.get("history", None)
    if hist is None:
        return []
    # NB: `hist` round-trips as a numpy array, so never use `or []` / truthiness on it.
    out = []
    for e in list(hist):
        if isinstance(e, (str, bytes)):
            try:
                e = json.loads(e)
            except Exception:
                continue
        if isinstance(e, dict):
            out.append(e)
    return out

In [ ]:
LINE       = "MDAMB468"
LINE_DISP  = "MDA-MB-468"

BASE       = Path("/Users/ronguy/Dropbox/Work/CyTOF/CyTOF_Christina")
DATA_CSV   = BASE / "Data/MDAMB/MDAMB468_Yael_cols_corrected.csv"
PLOTS      = BASE / "Plots"; PLOTS.mkdir(exist_ok=True)

REG          = Path("/Users/ronguy/Dropbox/Work/CyTOF/Code/CyTOFSTD/cytof_marker_registry_files")
PROJECT_PATH = Path(f"/Users/ronguy/Dropbox/Work/CyTOF/Projects/{LINE}_NormCompare")
RUN_ID       = "MDAMB468"

In [ ]:
ARCSINH_COFACTOR = 5.0
SEED             = 42
REBUILD          = False   # set True to wipe the project and re-run everything from scratch

# ---- the three normalization variants -------------------------------------------------
# key -> (display label, colour, kwargs for normalize_with_cytof_transform)
METHODS = {
    "divide":  ("divide (B1)",            "#3f78c1", dict(method="divide")),
    "single":  ("regress, single γ",      "#33a02c", dict(method="regress", gamma_mode="single")),
    "regress": ("regress, per-marker γ",  "#e2725b", dict(method="regress", gamma_mode="per_marker")),
}
MKEYS  = list(METHODS)
LABEL  = {k: v[0] for k, v in METHODS.items()}
COLOR  = {k: v[1] for k, v in METHODS.items()}

LAYER_OF = {k: f"norm_{k}"    for k in MKEYS}   # corrected, arcsinh scale
ZLAYER_OF= {k: f"norm_{k}_z"  for k in MKEYS}   # z from cytof_transform (corrected markers only)
ZS_OF    = {k: f"norm_{k}_zs" for k in MKEYS}   # explicit z-score step (section 5.3)
EMB_OF   = {k: f"umap_{k}"   for k in MKEYS}
CLKEY_OF = {k: f"cl_{k}"     for k in MKEYS}
TF_OF    = {k: f"tf_{k}"     for k in MKEYS}

## 1. Raw data inspection

In [ ]:
raw_df = pd.read_csv(DATA_CSV)
print(f"{raw_df.shape[0]:,} cells x {raw_df.shape[1]} markers")
print(f"NaNs: {raw_df.isna().sum().sum()}   negatives: {(raw_df < 0).sum().sum()}   zeros: {(raw_df == 0).sum().sum():,}")
raw_df.describe().T[['min', '25%', '50%', '75%', 'max']].round(2)

Values are large positive intensities with a hard floor at 0 and no negatives — this is **raw, pre-arcsinh**
data, which is exactly what the `divide` method requires. There are no DNA, Event-length or bead channels and
only one sample, so classical doublet/bead QC is not available here (and per instruction was done upstream).

## 2. Ingest into a CyTOFSTD project

In [ ]:
if REBUILD and PROJECT_PATH.exists():
    shutil.rmtree(PROJECT_PATH)

try:
    project = Project.load(str(PROJECT_PATH))
    print(f"Loaded existing project at {PROJECT_PATH}")
except Exception:
    PROJECT_PATH.parent.mkdir(parents=True, exist_ok=True)
    project = Project.create(
        str(PROJECT_PATH),
        project_id=f"{LINE}_NormCompare",
        project_name=f"{LINE_DISP} normalization method comparison",
        standard_marker_file=str(REG / "standard_markers.csv"),
        marker_alias_file=str(REG / "marker_aliases.yaml"),
    )
    print(f"Created project at {PROJECT_PATH}")

In [ ]:
# cytofstandard's ingest needs a sample-metadata table with file_name / sample_id / line_id.
PREP = PROJECT_PATH / "preprocessed"; PREP.mkdir(parents=True, exist_ok=True)
meta_path = PREP / "samples.csv"
pd.DataFrame([{
    "file_name": DATA_CSV.name,
    "sample_id": RUN_ID,
    "line_id":   RUN_ID,
    "condition": "baseline",
}]).to_csv(meta_path, index=False)

if project.has_run(RUN_ID):
    run = project.get_run(RUN_ID)
    print(f"Run '{RUN_ID}' already exists — skipping ingestion.")
else:
    run = project.add_run(RUN_ID, run_name=f"{LINE_DISP} (cols corrected)")
    run.ingest(
        files=[str(DATA_CSV)],
        sample_metadata=str(meta_path),
        strict_markers=False,
        allow_extra_markers=True,
        show_marker_coverage=True,
    )
    print("Ingested.")

adata = run.read_adata()
print(f"\nAnnData: {adata.n_obs:,} cells x {adata.n_vars} markers")
print("X is raw (identical to layers['raw']):", np.allclose(adata.X, adata.layers['raw']))

In [ ]:
# All 32 channels resolve exactly against the standard marker registry.
run.ingestion_summary()

## 3. QC — core histones > 5

The only QC applied: a cell must have `H3 > 5`, `H4 > 5` and `H3.3 > 5` on raw data.
`qc_gate` bounds are inclusive (`>=`), but no cell sits exactly at 5, so `lower=5` is identical to `> 5` here.

In [ ]:
CORE_HISTONES = run.markers_core_histones           # from the registry: is_core_histone == True
print("core histones (registry):", CORE_HISTONES)

pre = run.read_adata()
raw_pre = pd.DataFrame(pre.layers["raw"], columns=pre.var_names, index=pre.obs_names)

for m in CORE_HISTONES:
    n_bad = int((raw_pre[m] <= 5).sum())
    print(f"  {m:6s}  <=5 : {n_bad:5d} cells ({100*n_bad/len(raw_pre):.2f}%)")
print(f"  exactly ==5 anywhere: {int((raw_pre[CORE_HISTONES] == 5).sum().sum())}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, m in zip(axes, CORE_HISTONES):
    ax.hist(np.log10(raw_pre[m] + 1), bins=200, color="0.35")
    ax.axvline(np.log10(6), color="crimson", lw=1.5, ls="--", label="gate (>5)")
    ax.set_title(m); ax.set_xlabel("log10(raw + 1)"); ax.set_yscale("log")
    ax.legend(fontsize=8)
axes[0].set_ylabel("cells")
fig.suptitle("QC gate on core histones (raw scale)", y=1.03)
plt.tight_layout(); plt.show()

In [ ]:
qc_hist = uns_history(run.read_adata(), "qc")
if qc_hist:
    h = qc_hist[-1]
    print(f"QC already applied ({len(qc_hist)} event(s)) — skipping to stay idempotent.")
    print(f"  n_before={h['n_before']:,}  n_after={h['n_after']:,}  n_removed={h['n_removed']:,}")
else:
    mask = run.qc_gate({m: {"lower": 5} for m in CORE_HISTONES}, layer="raw", inplace=True)
    print(f"QC: kept {int(mask.sum()):,} / {len(mask):,} cells  (removed {int((~mask).sum()):,}, "
          f"{100*(~mask).mean():.2f}%)")

adata = run.read_adata()
print(f"\nPost-QC: {adata.n_obs:,} cells x {adata.n_vars} markers")

## 4. Marker groups

Permeabilization is a property of getting antibody **inside** the cell, so only **intracellular** markers are
corrected. The four extracellular markers (`CD24`, `CD44`, `CD49f`, `EpCAM`, taken from the registry's
`intra_extra` field) are deliberately left out of `markers_to_correct`.

This matters for two reasons:

1. **Correctness.** A surface marker has no mechanistic dependence on permeabilization. Applying a correction
   to it can only *inject* the technical factor into a channel that never carried it.
2. **They become a built-in negative control.** Because they pass through untouched, any residual correlation
   between a surface marker and `tech_ref` after normalization is, by construction, **not** something
   normalization failed to remove — it is evidence that `tech_ref` itself carries biology (cell size, ploidy,
   cell-cycle stage). That is used in section 9 to calibrate how much of the "technical" axis is really
   technical.

In [ ]:
ALL_MARKERS = run.read_adata().var_names.tolist()

# 16 histone post-translational modifications -> drive UMAP + clustering.
# EZH2 is registry class "Chromatin" but is a writer enzyme, not a modification, so it is
# excluded from the clustering feature set (kept for plotting).
EPI = [m for m in ['H2AK119ub','H3K27ac','H3K27me2','H3K27me3','H3K36me2','H3K36me3',
                   'H3K4me1','H3K4me3','H3K64ac','H3K9ac','H3K9me2','H3K9me3',
                   'H3S28p','H4K16ac','H4K20me3','pH2A.X'] if m in ALL_MARKERS]

# Permeabilization affects INTRACELLULAR staining. Surface markers have no mechanistic
# reason to depend on it, so they are excluded from the correction entirely — correcting
# them can only inject the technical factor into channels that never carried it.
SURFACE = [m for m in run.markers_extracellular if m in ALL_MARKERS]
INTRA   = [m for m in run.markers_intracellular if m in ALL_MARKERS]

TO_CORRECT   = [m for m in INTRA if m not in CORE_HISTONES]   # 25 intracellular markers
INTRA_OTHER  = [m for m in TO_CORRECT if m not in EPI]        # intracellular, non-epigenetic

# post-normalization marker universe (core histones dropped, per instruction)
MARKERS_POST = [m for m in ALL_MARKERS if m not in CORE_HISTONES]
OTHER = [m for m in MARKERS_POST if m not in EPI]

# marker -> role, used to group every downstream metric
ROLE = {m: ("epigenetic" if m in EPI else
            "intracellular (other)" if m in INTRA_OTHER else
            "surface (UNcorrected)") for m in MARKERS_POST}

print(f"CORE_HISTONES ({len(CORE_HISTONES):2d}) control markers   : {CORE_HISTONES}")
print(f"TO_CORRECT    ({len(TO_CORRECT):2d}) corrected          : intracellular only")
print(f"  EPI         ({len(EPI):2d}) -> UMAP/clustering : {EPI}")
print(f"  INTRA_OTHER ({len(INTRA_OTHER):2d})                    : {INTRA_OTHER}")
print(f"SURFACE       ({len(SURFACE):2d}) NOT corrected      : {SURFACE}")

## 5. Normalization — three variants

All three start from the **same** post-QC `raw` layer and correct the **same** 29 markers, writing to separate
layers so nothing is overwritten.

`normalize_with_cytof_transform` writes the tech factor to `obs["norm_tech_factor"]` on every call, so it is
copied to a method-specific column immediately after each one.

Note on the fourth available mode: `gamma_mode="shrink"` is **not** included, and deliberately. Its shrinkage
weight is `B = τ²/(τ² + SE²)`; with ~67k cells `SE → 0`, so `B → 1` and it collapses back onto `per_marker`,
adding a redundant arm. `"shrink_stability"` would need a stability grouping variable, which a single sample
does not provide.

In [ ]:
norm_hist  = uns_history(run.read_adata(), "normalization")
# key on corrected_layer, NOT on method: two of the three arms share method="regress"
done_layers = {h.get("corrected_layer") for h in norm_hist}
print("normalized layers already present:", done_layers or "none")

In [ ]:
summaries = {}
for k in MKEYS:
    layer = LAYER_OF[k]
    if layer in done_layers:
        summaries[k] = [h for h in norm_hist if h.get("corrected_layer") == layer][-1]
        print(f"{LABEL[k]:24s} : already applied — skipping.")
        continue
    print(f"{LABEL[k]:24s} : running ...")
    summaries[k] = run.normalize_with_cytof_transform(
        control_markers=CORE_HISTONES,
        markers_to_correct=TO_CORRECT,
        source_layer="raw",
        input_is_arcsinh=False,          # divide: raw->divide->arcsinh; regress: wrapper arcsinhs first
        arcsinh_cofactor=ARCSINH_COFACTOR,
        groupby_col="sample_id",
        corrected_layer=layer,
        z_layer=ZLAYER_OF[k],
        anchor_to_median=True,
        zscore=True,
        **METHODS[k][2],
    )
    a = run.read_adata(); a.obs[TF_OF[k]] = a.obs["norm_tech_factor"].values; run.save(a)

pd.DataFrame({k: {"method": s["method"], "gamma_mode": s["gamma_mode"],
                  "tech_factor": s["tech_factor_kind"], "layer": s["corrected_layer"]}
              for k, s in summaries.items()}).T

In [ ]:
adata = run.read_adata()

raw_all   = pd.DataFrame(adata.layers["raw"], columns=adata.var_names, index=adata.obs_names)
asinh_raw = np.arcsinh(raw_all / ARCSINH_COFACTOR)          # uncorrected reference, arcsinh scale

def layer_df(layer, markers=MARKERS_POST):
    M = pd.DataFrame(adata.layers[layer], columns=adata.var_names, index=adata.obs_names)
    return M[markers]

NORM = {k: layer_df(LAYER_OF[k]) for k in MKEYS}    # all markers, core histones dropped
# NORMz (the z-scored epigenetic matrix used for UMAP/clustering) is built in section 5.1,
# after the explicit z-scoring pass.

print("layers:", sorted(adata.layers.keys()))
print()
print(f"{'':26s} {'min':>8s} {'max':>8s} {'mean':>8s}")
print(f"{'raw (arcsinh)':26s} {asinh_raw[MARKERS_POST].values.min():8.3f} "
      f"{asinh_raw[MARKERS_POST].values.max():8.3f} {asinh_raw[MARKERS_POST].values.mean():8.3f}")
for k in MKEYS:
    M = NORM[k].values
    print(f"{LABEL[k]:26s} {M.min():8.3f} {M.max():8.3f} {M.mean():8.3f}")

### 5.1 Explicit z-scoring of every method

`cytof_transform` already emits a z-scored layer (`zscore=True`), but it only z-scores the markers listed in
`markers_to_correct` and does it inside the transform. An explicit pass is applied here instead so that the
step is visible, provenance-logged, and **identical across all three methods** — every marker is centred and
scaled the same way, so nothing downstream can be attributed to a difference in scaling.

`zscore_markers_balanced` z-scores within `groupby_col` on a group-balanced subsample; with a single sample
this reduces to a plain per-marker z-score, but it is the right call to use for provenance and it generalizes
if more samples are added later.

Note the shipped default `source_layer="normlized"` is misspelled in the package, so it is always passed
explicitly here.

In [ ]:
adata = run.read_adata()
for k in MKEYS:
    if ZS_OF[k] in adata.layers:
        print(f"{LABEL[k]:24s} : '{ZS_OF[k]}' exists — skipping.")
        continue
    run.zscore_markers_balanced(
        source_layer=LAYER_OF[k],      # NB: package default is misspelled "normlized"
        output_layer=ZS_OF[k],
        groupby_col="sample_id",
        random_state=SEED,
    )
    print(f"{LABEL[k]:24s} : z-scored {LAYER_OF[k]} -> {ZS_OF[k]}")

adata = run.read_adata()
chk = pd.DataFrame({
    LABEL[k]: pd.DataFrame(adata.layers[ZS_OF[k]], columns=adata.var_names)[EPI]
              .agg(['mean', 'std']).T.stack()
    for k in MKEYS
}).round(4)
print("\nper-marker mean/std of the epigenetic marks after z-scoring (should be ~0 / ~1):")
chk.groupby(level=1).agg(['min', 'max']).round(4)

In [ ]:
# Explicitly z-scored frames.
#   NORMZS : all markers (core histones dropped) -> used for the UMAP marker plots
#   NORMz  : epigenetic marks only               -> drives UMAP + Leiden
NORMZS = {k: pd.DataFrame(adata.layers[ZS_OF[k]], columns=adata.var_names,
                          index=adata.obs_names)[MARKERS_POST] for k in MKEYS}
NORMz  = {k: NORMZS[k][EPI] for k in MKEYS}
print({k: (NORMZS[k].shape, NORMz[k].shape) for k in MKEYS})

### 5.2 Technical factors

In [ ]:
tf = {k: adata.obs[TF_OF[k]].astype(float) for k in MKEYS}
tech_ref = asinh_raw[CORE_HISTONES].mean(axis=1)   # method-agnostic: mean arcsinh core histone

fig, axes = plt.subplots(1, len(MKEYS), figsize=(4.6 * len(MKEYS), 3.3))
for ax, k in zip(np.atleast_1d(axes).ravel(), MKEYS):
    ax.hist(tf[k], bins=200, color=COLOR[k])
    ax.axvline(np.median(tf[k]), color="k", ls="--", lw=1)
    ax.set_title(f"{LABEL[k]}\n({summaries[k]['tech_factor_kind']})", fontsize=10)
    ax.set_ylabel("cells")
plt.tight_layout(); plt.show()

print("Spearman rho against the method-agnostic reference (mean arcsinh core histone):")
for k in MKEYS:
    print(f"  {LABEL[k]:26s} {ss.spearmanr(tf[k], tech_ref).statistic:+.3f}")
print()
print("Between technical factors:")
for i, a_ in enumerate(MKEYS):
    for b_ in MKEYS[i+1:]:
        print(f"  {LABEL[a_]:26s} vs {LABEL[b_]:26s} {ss.spearmanr(tf[a_], tf[b_]).statistic:+.3f}")

The two `regress` arms share an identical technical factor by construction (both use PC1 of the core histones —
only the slopes applied to it differ), so their correlation is exactly 1. The interesting number is `divide`'s
`M` against the regress `f`: if that is high, all three are chasing the same technical axis and differ only in
how they remove it.

### 5.3 Gamma: per-marker vs single

In [ ]:
def gammas(summary):
    g = summary.get("gamma_by_group", {}) or {}
    if not g: return pd.Series(dtype=float)
    g = g.get(RUN_ID, g[list(g)[0]])
    return pd.Series(g, dtype=float)

g_per = gammas(summaries["regress"]).reindex(TO_CORRECT).dropna().sort_values()
g_one = gammas(summaries["single"]).reindex(TO_CORRECT).dropna()

# Fallback: the provenance history does not always survive a zarr round-trip, so if the
# stored gammas are missing, refit them exactly as `regress` does (OLS on PC1 of controls).
if len(g_per) == 0 or len(g_one) == 0:
    from sklearn.decomposition import PCA
    f_pc1 = PCA(n_components=1).fit_transform(asinh_raw[CORE_HISTONES].values)[:, 0]
    fit = pd.Series({m: np.polyfit(f_pc1, asinh_raw[m].values, 1)[0] for m in TO_CORRECT})
    if len(g_per) == 0:
        g_per = fit.sort_values()
        print("[gamma] per-marker γ refitted from data (provenance unavailable)")
    if len(g_one) == 0:
        g_one = pd.Series(fit.mean(), index=TO_CORRECT)
        print("[gamma] single γ̄ approximated as the mean OLS slope")

gbar = float(g_one.iloc[0]) if len(g_one) else np.nan

fig, ax = plt.subplots(figsize=(12, 3.9))
cols = [COLOR["regress"] if m in EPI else "0.65" for m in g_per.index]
ax.bar(range(len(g_per)), g_per.values, color=cols, label="per-marker γ")
ax.axhline(gbar, color=COLOR["single"], lw=2, ls="--", label=f"single γ̄ = {gbar:.3f}")
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(range(len(g_per))); ax.set_xticklabels(g_per.index, rotation=90)
ax.set_ylabel(r"$\gamma_m$"); ax.legend()
ax.set_title(r"regress: per-marker $\gamma_m$ vs the single shared $\bar{\gamma}$   (blue = epigenetic, grey = other)")
plt.tight_layout(); plt.show()

print(f"single γ̄ = {gbar:.4f}   (identical for all {len(g_one)} corrected markers)")
print(f"per-marker γ: min={g_per.min():.3f}  median={g_per.median():.3f}  max={g_per.max():.3f}")
print(f"markers whose per-marker γ differs from γ̄ by >50%: "
      f"{int((abs(g_per - gbar) > 0.5*abs(gbar)).sum())} / {len(g_per)}")
print("\nThe spread of the bars around the dashed line is exactly what the single-γ arm throws away,")
print("and exactly what the divide method also cannot represent.")

This plot is the crux of the comparison. `divide` and `regress single-γ` both apply **one** correction strength
to every marker — the dashed line. `regress per-marker γ` lets each marker have its own. Markers whose bar sits
far from the dashed line are the ones the two uniform methods necessarily get wrong: under-corrected where
`γ_m > γ̄`, over-corrected where `γ_m < γ̄`, and actively damaged where `γ_m ≈ 0` but a full correction is applied
anyway.

## 6. UMAP and clustering on the epigenetic modifications

UMAP and Leiden use the **16 epigenetic marks only**, from each method's z-scored layer (all three z-score the
same 16 markers, so the inputs are on a comparable footing).

In [ ]:
UMAP_KW = dict(n_neighbors=15, min_dist=0.1, metric="euclidean", random_state=SEED)
RESOLUTION = 1.0

existing_emb = set((run.read_adata().uns.get("embeddings", {}) or {}).keys())
for k in MKEYS:
    if EMB_OF[k] in existing_emb:
        print(f"{LABEL[k]:24s} : embedding exists — skipping UMAP.")
    else:
        print(f"{LABEL[k]:24s} : computing UMAP on {len(EPI)} epigenetic marks ...")
        run.compute_umap(markers=EPI, source_layer=ZS_OF[k],
                         embedding_name=EMB_OF[k], **UMAP_KW)
    run.cluster_leiden(embedding_name=EMB_OF[k], cluster_key=CLKEY_OF[k],
                       resolution=RESOLUTION, seed=SEED)

adata = run.read_adata()
for k in MKEYS:
    print(f"{LABEL[k]:24s} -> {adata.obs[CLKEY_OF[k]].nunique()} Leiden clusters")

In [ ]:
adata = run.read_adata()
U  = {k: np.asarray(adata.obsm[EMB_OF[k]])  for k in MKEYS}
CL = {k: adata.obs[CLKEY_OF[k]].astype(str) for k in MKEYS}

# scanpy resolves `basis` against obsm directly, so EMB_OF names work as-is.
# Cluster keys must be categorical for a discrete palette.
for k in MKEYS:
    adata.obs[CLKEY_OF[k]] = adata.obs[CLKEY_OF[k]].astype(str).astype("category")
adata.obs["tech_ref"] = tech_ref.values      # method-agnostic technical axis, for 6.1

fig, axes = plt.subplots(1, len(MKEYS), figsize=(6.3 * len(MKEYS), 5.8))
for ax, k in zip(np.atleast_1d(axes).ravel(), MKEYS):
    sc.pl.embedding(
        adata, basis=EMB_OF[k], color=CLKEY_OF[k], ax=ax, show=False,
        legend_loc="on data", legend_fontsize=9, legend_fontoutline=2,
        palette="tab20", size=3, frameon=False, use_raw=False,
        title=f"{LABEL[k]} — {CL[k].nunique()} clusters",
    )
fig.suptitle("UMAP on the 16 epigenetic modifications", y=1.02, fontsize=13)
plt.tight_layout(); plt.savefig(PLOTS / f"{LINE}_norm_umap_clusters.png", dpi=180, bbox_inches="tight"); plt.show()

### 6.1 Technical factor on the UMAPs

A well-normalized embedding should **not** be organized by the technical factor. Structure mirroring `tech_ref`
here means residual permeabilization signal is still driving the clustering.

In [ ]:
fig, axes = plt.subplots(1, len(MKEYS), figsize=(6.3 * len(MKEYS), 5.2))
for ax, k in zip(np.atleast_1d(axes).ravel(), MKEYS):
    sc.pl.embedding(
        adata, basis=EMB_OF[k], color="tech_ref", ax=ax, show=False,
        cmap="viridis", vmin="p1", vmax="p99", size=3, frameon=False,
        use_raw=False, title=LABEL[k],
    )
fig.suptitle("Coloured by mean arcsinh core histone (the technical axis)", y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

print(f"{'':26s} {'|rho| UMAP1':>12s} {'|rho| UMAP2':>12s}   (lower = less technical structure)")
for k in MKEYS:
    r1 = abs(ss.spearmanr(U[k][:, 0], tech_ref).statistic)
    r2 = abs(ss.spearmanr(U[k][:, 1], tech_ref).statistic)
    print(f"{LABEL[k]:26s} {r1:12.3f} {r2:12.3f}")

## 7. All markers on every UMAP

In [ ]:
def marker_grid(k, markers, ncols=5, fname=None):
    """UMAP coloured by each z-scored marker, via scanpy, with large bold visible text."""
    nrows = int(np.ceil(len(markers) / ncols))
    fig, axes_arr = plt.subplots(nrows, ncols, figsize=(ncols * 3.8, nrows * 3.6))
    axes_flat = axes_arr.ravel()

    for idx, mname in enumerate(markers):
        ax = axes_flat[idx]
        sc.pl.embedding(
            adata, basis=EMB_OF[k], color=mname, layer=ZS_OF[k],
            cmap="seismic", vcenter=0, vmin="p1", vmax="p99",
            ax=ax, size=4, frameon=False, use_raw=False,
            colorbar_loc="right", show=False
        )
        ax.set_title(mname, fontsize=13, fontweight="bold", pad=6)

    for ax in axes_flat[len(markers):]:
        ax.axis("off")

    fig.suptitle(f"{LINE_DISP} — {LABEL[k]} — All Markers (Z-Scored, Seismic)",
                 fontsize=18, fontweight="bold", y=1.02)
    plt.tight_layout()
    if fname:
        fig.savefig(PLOTS / fname, dpi=200, bbox_inches="tight")
    plt.show()

ALL_MARKERS_SORTED = sorted(EPI + OTHER, key=lambda s: s.lower())
marker_grid("divide", ALL_MARKERS_SORTED, fname=f"{LINE}_umap_divide_allmarkers.png")

In [ ]:
marker_grid("single", ALL_MARKERS_SORTED, fname=f"{LINE}_umap_single_allmarkers.png")

In [ ]:
marker_grid("regress", ALL_MARKERS_SORTED, fname=f"{LINE}_umap_regress_allmarkers.png")

### 7.1 Cluster marker profiles

In [ ]:
fig, axes = plt.subplots(1, len(MKEYS), figsize=(7 * len(MKEYS), 6.5))
for ax, k in zip(np.atleast_1d(axes).ravel(), MKEYS):
    prof = NORM[k][EPI + OTHER].groupby(CL[k].values).mean()
    prof.index = [f"c{i}" for i in prof.index]
    z = (prof - prof.mean()) / prof.std().replace(0, 1)
    sns.heatmap(z.T, cmap="RdBu_r", center=0, ax=ax,
                cbar_kws={"shrink": .6, "label": "z (across clusters)"})
    ax.set_title(f"{LABEL[k]}"); ax.set_xlabel("Leiden cluster")
fig.suptitle("Cluster mean marker profiles", y=1.01, fontsize=13)
plt.tight_layout(); plt.show()

## 8. Clustering agreement between the three methods

In [ ]:
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             adjusted_mutual_info_score, fowlkes_mallows_score)

pairs = [(a_, b_) for i, a_ in enumerate(MKEYS) for b_ in MKEYS[i+1:]]
rows = []
for a_, b_ in pairs:
    x, y = CL[a_].values, CL[b_].values
    rows.append({
        "pair": f"{LABEL[a_]}  vs  {LABEL[b_]}",
        "ARI": adjusted_rand_score(x, y),
        "NMI": normalized_mutual_info_score(x, y),
        "AMI": adjusted_mutual_info_score(x, y),
        "FMI": fowlkes_mallows_score(x, y),
    })
agree = pd.DataFrame(rows).set_index("pair").round(4)
print("clusters: " + "  ".join(f"{LABEL[k]}={CL[k].nunique()}" for k in MKEYS))
agree

In [ ]:
# ARI / NMI matrices
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, metric, fn in zip(axes, ["ARI", "NMI"],
                          [adjusted_rand_score, normalized_mutual_info_score]):
    M = pd.DataFrame([[fn(CL[a_].values, CL[b_].values) for b_ in MKEYS] for a_ in MKEYS],
                     index=[LABEL[k] for k in MKEYS], columns=[LABEL[k] for k in MKEYS])
    sns.heatmap(M, annot=True, fmt=".3f", cmap="viridis", vmin=0, vmax=1, ax=ax,
                cbar_kws={"shrink": .8})
    ax.set_title(metric)
plt.tight_layout(); plt.show()

In [ ]:
# Detailed correspondence via cytofstandard's hypergeometric cluster matcher.
match = {}
for a_, b_ in pairs:
    match[(a_, b_)] = run.match_clusterings(CLKEY_OF[a_], CLKEY_OF[b_],
                                            score_mode="jaccard", n_permutations=200,
                                            random_state=SEED, plot=False)

ncol = 3; nrow = int(np.ceil(len(pairs) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(6.3 * ncol, 5 * nrow))
axes = np.atleast_1d(axes).ravel()
for ax in axes[len(pairs):]:
    ax.axis("off")
for ax, (a_, b_) in zip(axes, pairs):
    ct = match[(a_, b_)]["contingency"]
    frac = ct.div(ct.sum(axis=1), axis=0)
    sns.heatmap(frac, cmap="Blues", ax=ax, cbar_kws={"shrink": .7, "label": "row fraction"})
    ax.set_xlabel(LABEL[b_]); ax.set_ylabel(LABEL[a_])
    ax.set_title(f"1:1 jaccard = {match[(a_, b_)]['global_one_to_one_score']:.3f}", fontsize=10)
fig.suptitle("Cluster correspondence (rows normalized)", y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

## 9. Which normalization is best?

Cluster agreement tells us the methods *differ*, not which is right. The criteria below are what normalization
is actually supposed to achieve.

**Reference used throughout:** `tech_ref` = mean arcsinh core-histone signal per cell, computed from the **raw**
data and therefore independent of all three methods.

### 9.1 Residual dependence on the technical factor (primary target)

> **Read this metric with care.** `regress` with `per_marker` γ fits an OLS slope against PC1 of the core
> histones and subtracts it, so driving this correlation to ~0 is literally its objective function. It is
> close to guaranteed to win here, and the single-γ arm partially shares that advantage. The metric still
> earns its place — it quantifies how much technical signal each method leaves behind — but it must **not**
> on its own decide the winner. Sections 9.2–9.4 carry the weight.

In [ ]:
def resid_corr(M, ref):
    return M.apply(lambda c: abs(ss.spearmanr(c.values, ref.values).statistic))

rc = pd.DataFrame({"raw (arcsinh)": resid_corr(asinh_raw[MARKERS_POST], tech_ref)})
for k in MKEYS:
    rc[LABEL[k]] = resid_corr(NORM[k], tech_ref)
rc["group"] = [ROLE[m] for m in rc.index]

cols_plot = ["raw (arcsinh)"] + [LABEL[k] for k in MKEYS]
colors_plot = ["0.6"] + [COLOR[k] for k in MKEYS]

fig, ax = plt.subplots(figsize=(12, 3.8))
order = rc.sort_values("raw (arcsinh)", ascending=False).index
x = np.arange(len(rc)); w = 0.8 / (len(MKEYS) + 1)
for i, (col, c) in enumerate(zip(cols_plot, colors_plot)):
    ax.bar(x + (i - len(MKEYS)/2) * w, rc.loc[order, col].values, w, label=col, color=c)
ax.set_xticks(x); ax.set_xticklabels(order, rotation=90)
ax.set_ylabel(r"|Spearman $\rho$| with tech_ref"); ax.legend(fontsize=9)
ax.set_title("Residual dependence on core-histone content (lower = better)")
plt.tight_layout(); plt.show()

print(rc.groupby("group")[cols_plot].mean().round(4).to_string())
print()
print("NOTE the 'surface (UNcorrected)' row: those markers were never touched, so their")
print("values are identical to raw by construction. Whatever correlation with tech_ref they")
print("retain is BIOLOGY carried by tech_ref (cell size / ploidy), not a normalization failure.")
print()
print("overall mean:")
print(rc[cols_plot].mean().round(4).to_string())

### 9.2 Over-correction: is biological signal being destroyed?

Removing technical variance is easy if you also remove the biology. This checks how much marker variance
survives, and whether the **biological correlation structure** between epigenetic marks is preserved.
Co-regulated marks (the H3K27me2/me3 axis, active marks H3K27ac/H3K9ac/H3K4me3) should stay correlated —
but a *uniform* correction applied to every marker will tend to **manufacture** shared variance and inflate
those correlations artificially.

In [ ]:
var_tbl = pd.DataFrame({"raw (arcsinh)": asinh_raw[MARKERS_POST].var()})
for k in MKEYS:
    var_tbl[LABEL[k]] = NORM[k].var()
ratio = var_tbl.div(var_tbl["raw (arcsinh)"], axis=0)[[LABEL[k] for k in MKEYS]]
ratio["group"] = [ROLE[m] for m in ratio.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
order = ratio.sort_values(LABEL["divide"]).index
x = np.arange(len(ratio)); w = 0.8 / len(MKEYS)
for i, k in enumerate(MKEYS):
    axes[0].bar(x + (i - (len(MKEYS)-1)/2) * w, ratio.loc[order, LABEL[k]], w, label=LABEL[k], color=COLOR[k])
axes[0].axhline(1, color="k", ls="--", lw=1)
axes[0].set_xticks(x); axes[0].set_xticklabels(order, rotation=90)
axes[0].set_ylabel("variance / raw variance"); axes[0].legend(fontsize=8)
axes[0].set_title("Variance retained (1.0 = unchanged)")

sns.boxplot(data=ratio.melt(id_vars="group", var_name="method", value_name="ratio"),
            x="method", y="ratio", hue="group", ax=axes[1])
axes[1].axhline(1, color="k", ls="--", lw=1)
axes[1].set_title("Variance retained by marker group")
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

print(ratio.groupby("group")[[LABEL[k] for k in MKEYS]].median().round(3).to_string())
print()
print("surface markers should read exactly 1.000 for all three methods (untouched) — a")
print("deviation would mean something leaked into the uncorrected channels.")

In [ ]:
# Preservation of the biological correlation structure among epigenetic marks.
def upper(M):
    c = M.corr(method="spearman").values
    return c[np.triu_indices_from(c, k=1)]

c_raw = upper(asinh_raw[EPI])
c_k   = {k: upper(NORM[k][EPI]) for k in MKEYS}

fig, axes = plt.subplots(1, len(MKEYS) + 1, figsize=(4.8 * (len(MKEYS) + 1), 4))
for ax, k in zip(np.atleast_1d(axes).ravel()[:len(MKEYS)], MKEYS):
    ax.scatter(c_raw, c_k[k], s=16, alpha=0.6, color=COLOR[k])
    ax.plot([-1, 1], [-1, 1], "k--", lw=1)
    ax.set_xlabel("raw (arcsinh) pairwise rho"); ax.set_ylabel(f"{LABEL[k]} rho")
    ax.set_title(f"{LABEL[k]}\nr={np.corrcoef(c_raw, c_k[k])[0,1]:.3f}, "
                 f"median shift={np.median(c_k[k] - c_raw):+.3f}", fontsize=10)
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
axes[-1].hist([c_raw] + [c_k[k] for k in MKEYS], bins=18,
             label=["raw"] + [LABEL[k] for k in MKEYS],
             color=["0.6"] + [COLOR[k] for k in MKEYS])
axes[-1].set_title("Distribution of pairwise correlations"); axes[3].legend(fontsize=8)
axes[-1].set_xlabel("Spearman rho between epigenetic marks")
plt.tight_layout(); plt.show()

print("Median shift in pairwise correlation between epigenetic marks vs raw:")
for k in MKEYS:
    print(f"  {LABEL[k]:26s} {np.median(c_k[k] - c_raw):+.4f}")
print("\nA large POSITIVE shift means the method INDUCED correlation between marks — the")
print("signature of applying one shared correction factor to every marker.")

### 9.3 Is the technical factor actually technical? (the confounding test)

Everything above assumes `tech_ref` is a nuisance axis. It is not purely one. Core-histone content per cell
also tracks **cell size, ploidy and cell-cycle stage** — a G2/M cell genuinely holds about twice the histone
of a G1 cell. To the extent that is true, regressing markers on `tech_ref` removes **biology**, and the
per-marker mode has the most freedom to do exactly that.

Two independent pieces of evidence bear on this:

1. **The uncorrected surface markers.** They were never touched, so their correlation with `tech_ref` is a
   direct readout of how much *biology* lives on that axis. A surface marker cannot be affected by nuclear
   permeabilization, so any correlation it shows is cell size / state.
2. **Does γ_m track raw correlation with `tech_ref`?** If it does, per-marker γ is not learning a
   marker-specific *technical* sensitivity — it is learning how biologically coupled each marker is to cell
   state, and subtracting that.

In [ ]:
# 1. Surface markers as a negative control
surf_rho = asinh_raw[SURFACE].corrwith(tech_ref, method="spearman").abs().round(3)
print("|Spearman rho| of UNCORRECTED surface markers with tech_ref (raw):")
print(surf_rho.to_string())
print(f"\nmean = {surf_rho.mean():.3f}")
print("\nThese channels cannot be affected by permeabilization, so this is the FLOOR of")
print("biological (cell size / ploidy / state) signal carried by tech_ref. Any method that")
print("drives corrected markers well below this floor is removing biology, not artifact.")

In [ ]:
# 2. Does gamma track biological coupling rather than a technical sensitivity?
gam_all = g_per.reindex(TO_CORRECT).dropna()
raw_rho = asinh_raw[gam_all.index].corrwith(tech_ref, method="spearman")
r_gam = np.corrcoef(gam_all.values, raw_rho.values)[0, 1]

fig, ax = plt.subplots(figsize=(5.6, 4.6))
for grp, mk, c in [("epigenetic", "o", COLOR["regress"]), ("intracellular (other)", "s", "0.4")]:
    sel = [m for m in gam_all.index if ROLE[m] == grp]
    ax.scatter(raw_rho[sel], gam_all[sel], marker=mk, s=45, alpha=0.8, color=c, label=grp)
for m in ["H3S28p", "ER", "GATA3", "KI67", "KRT5", "H3K4me3"]:
    if m in gam_all.index:
        ax.annotate(m, (raw_rho[m], gam_all[m]), fontsize=8,
                    xytext=(4, 3), textcoords="offset points")
ax.axhline(gbar, color=COLOR["single"], ls="--", lw=1.5, label=f"single γ̄ = {gbar:.2f}")
ax.set_xlabel(r"raw Spearman $\rho$ with tech_ref"); ax.set_ylabel(r"fitted $\gamma_m$")
ax.set_title(f"γ tracks biological coupling to tech_ref\nr = {r_gam:.3f}", fontsize=11)
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"corr(gamma_m, raw rho with tech_ref) = {r_gam:.3f}")
print("\nThe more a marker covaries with cell histone content — for ANY reason, technical or")
print("biological — the larger the slope per-marker γ subtracts from it. The method cannot")
print("distinguish the two, so markers that are biologically coupled to cell state (H3S28p,")
print("ER, GATA3, KI67) are corrected hardest.")

### 9.4 Direct test: does known biology survive?

The metrics so far are proxies. These are direct: biological relationships that are known to be real in this
system, measured before and after each normalization. `tech_ref` is not involved.

In [ ]:
# Biological populations / relationships defined on RAW data, independent of any method.
bio_pairs = [("KI67", "H3S28p", "proliferation axis"),
             ("ER", "GATA3", "luminal lineage TFs"),
             ("H3K27me3", "H3K27me2", "PRC2 methylation axis"),
             ("H3K27ac", "H3K9ac", "active acetylation axis")]
bio_pairs = [(x, y, lbl) for x, y, lbl in bio_pairs
             if x in MARKERS_POST and y in MARKERS_POST]

rows = []
for x, y, lbl in bio_pairs:
    row = {"pair": f"{x} ~ {y}", "biology": lbl,
           "raw": ss.spearmanr(asinh_raw[x], asinh_raw[y]).statistic}
    for k in MKEYS:
        row[LABEL[k]] = ss.spearmanr(NORM[k][x], NORM[k][y]).statistic
    rows.append(row)
biocorr = pd.DataFrame(rows).set_index("pair").round(3)
print("Correlation between markers that are genuinely co-regulated (higher = biology kept):")
biocorr

In [ ]:
# Mitotic population, defined by RAW H3S28p, then checked in each normalized space.
mit = asinh_raw["H3S28p"] > asinh_raw["H3S28p"].quantile(0.99)
print(f"mitotic gate (raw H3S28p top 1%): {int(mit.sum())} cells\n")

rows = []
for k in MKEYS:
    v = NORM[k]["H3S28p"]
    auc = ss.mannwhitneyu(v[mit], v[~mit]).statistic / (mit.sum() * (~mit).sum())
    eff = (v[mit].mean() - v[~mit].mean()) / v.std()
    rows.append({"method": LABEL[k], "AUC": auc, "effect size (sd)": eff,
                 "H3S28p var retained": NORM[k]["H3S28p"].var() / asinh_raw["H3S28p"].var()})
print(pd.DataFrame(rows).set_index("method").round(3).to_string())

In [ ]:
# Variance retained for markers that carry the main biological axes of this system.
BIO = [m for m in ["H3S28p", "KI67", "ER", "GATA3", "ZEB1", "Vimentin",
                   "KRT5", "KRT8-18", "p53", "EZH2"] if m in MARKERS_POST]
vr = pd.DataFrame({LABEL[k]: (NORM[k][BIO].var() / asinh_raw[BIO].var()) for k in MKEYS}).round(3)

fig, ax = plt.subplots(figsize=(9, 3.6))
x = np.arange(len(BIO)); w = 0.8 / len(MKEYS)
for i, k in enumerate(MKEYS):
    ax.bar(x + (i - (len(MKEYS)-1)/2) * w, vr.loc[BIO, LABEL[k]], w, label=LABEL[k], color=COLOR[k])
ax.axhline(1, color="k", ls="--", lw=1)
ax.set_xticks(x); ax.set_xticklabels(BIO, rotation=45, ha="right")
ax.set_ylabel("variance / raw"); ax.legend(fontsize=8)
ax.set_title("Variance retained by biology-carrying markers (1.0 = untouched)")
plt.tight_layout(); plt.show()

print(vr.to_string()); print("\nmedian:"); print(vr.median().round(3).to_string())

### 9.5 Cluster quality in the epigenetic space

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

rng = np.random.default_rng(SEED)
n_sub = min(10000, len(NORMz[MKEYS[0]]))
idx = rng.choice(len(NORMz[MKEYS[0]]), size=n_sub, replace=False)

qual = pd.DataFrame([{
    "method": LABEL[k],
    "n_clusters": CL[k].nunique(),
    "silhouette":        silhouette_score(NORMz[k].values[idx], CL[k].values[idx]),
    "calinski_harabasz": calinski_harabasz_score(NORMz[k].values[idx], CL[k].values[idx]),
    "davies_bouldin":    davies_bouldin_score(NORMz[k].values[idx], CL[k].values[idx]),
} for k in MKEYS]).set_index("method").round(3)
print(f"computed on {n_sub:,} subsampled cells")
print("silhouette / calinski-harabasz: higher = better;  davies-bouldin: lower = better")
qual

### 9.6 Summary scorecard

In [ ]:
score = pd.DataFrame({
    LABEL[k]: {
        "mean |rho| with tech_ref (all markers)": rc[LABEL[k]].mean(),
        "mean |rho| with tech_ref (epigenetic)":  rc.loc[rc.group == "epigenetic", LABEL[k]].mean(),
        "median variance retained (epigenetic)":  ratio.loc[ratio.group == "epigenetic", LABEL[k]].median(),
        "median shift in pairwise mark corr":     np.median(c_k[k] - c_raw),
        "|rho| tech_ref vs UMAP1":                abs(ss.spearmanr(U[k][:, 0], tech_ref).statistic),
        "|rho| tech_ref vs UMAP2":                abs(ss.spearmanr(U[k][:, 1], tech_ref).statistic),
        "silhouette (epigenetic space)":          qual.loc[LABEL[k], "silhouette"],
        "davies_bouldin (lower better)":          qual.loc[LABEL[k], "davies_bouldin"],
        "n clusters":                             qual.loc[LABEL[k], "n_clusters"],
    } for k in MKEYS
}).round(4)
score

### How to read the scorecard

| row | direction | what it means |
|---|---|---|
| mean \|rho\| with tech_ref | **lower** | residual permeabilization signal left in the markers. The regress arms optimize this directly — see the caveat in 9.1. |
| variance retained (epigenetic) | **near 1** | far below 1 means biology was scrubbed out along with the technical factor |
| median shift in pairwise mark corr | **near 0** | strongly positive = the method *manufactured* correlation between marks by applying one shared factor to all of them |
| \|rho\| tech_ref vs UMAP1/2 | **lower** | the embedding is still organized by the technical axis |
| silhouette | **higher** | crisper, better separated clusters in the epigenetic space |
| davies_bouldin | **lower** | same idea, inverted |

**The three-arm design makes the diagnosis sharper than a two-way test could.** Read it as two contrasts:

1. **divide vs regress single-γ** — both apply one uniform correction, so any gap between them is attributable
   to the *form* of the correction (multiplicative on raw vs additive in arcsinh), not to per-marker tuning.
2. **regress single-γ vs regress per-marker γ** — identical machinery apart from the slopes, so any gap is the
   value of letting each marker have its own sensitivity. Section 5.2 predicts how large this should be: the
   wider the spread of γ_m around γ̄, the more the uniform methods must be leaving on the table.

A method wins only if it suppresses the technical axis **while** keeping variance near 1 and **without**
inflating inter-mark correlation. Winning on residual correlation alone, by flattening everything, is not a win.

### 9.7 Verdict

The verdict below is **computed from this cell line's own numbers**, not copied from another line. The
interpretive framework is fixed; the values, flags and recommendation are derived at run time so the four
notebooks can be compared directly.

The single most important number is the **surface-marker floor**. The 4 surface markers are never corrected,
so their correlation with `tech_ref` measures how much *biology* (cell size, ploidy, cell-cycle stage) lives
on that axis. Driving corrected markers far *below* that floor is not a better correction — it is removing
biology. This inverts the naive reading of section 9.1, where lower always looked better.

In [ ]:
FLOOR = float(surf_rho.mean())          # biological floor from untouched surface markers
epi_resid = {k: rc.loc[rc.group == "epigenetic", LABEL[k]].mean() for k in MKEYS}
oth_resid = {k: rc.loc[rc.group == "intracellular (other)", LABEL[k]].mean() for k in MKEYS}
raw_epi   = rc.loc[rc.group == "epigenetic", "raw (arcsinh)"].mean()

print(f"=== {LINE_DISP} — verdict inputs " + "=" * 40)
print(f"\nBiological floor (uncorrected surface markers): {FLOOR:.3f}")
print(f"Raw epigenetic residual (before any correction): {raw_epi:.3f}\n")

print(f"{'method':26s} {'epi |rho|':>10s} {'other |rho|':>12s} {'vs floor':>10s}  flag")
for k in MKEYS:
    frac = epi_resid[k] / FLOOR
    flag = ("OVERSHOOT (below floor)" if frac < 0.75 else
            "under-corrected"        if epi_resid[k] > 0.75 * raw_epi else
            "lands near the floor")
    print(f"{LABEL[k]:26s} {epi_resid[k]:10.3f} {oth_resid[k]:12.3f} {frac:9.2f}x  {flag}")

print("\n'vs floor' is epigenetic residual / surface floor. Near 1.0 = the correction stopped")
print("where biology begins. Well below ~0.75 = it kept going and took biology with it.")

In [ ]:
# Biology preservation: which arm keeps each known relationship best?
print("Known biological relationships (Spearman rho) — 'best' = closest to raw:\n")
best_bio = {}
for pair in biocorr.index:
    raw_v = biocorr.loc[pair, "raw"]
    devs  = {k: abs(biocorr.loc[pair, LABEL[k]] - raw_v) for k in MKEYS}
    win   = min(devs, key=devs.get)
    best_bio[pair] = win
    vals = "  ".join(f"{LABEL[k]}={biocorr.loc[pair, LABEL[k]]:+.3f}" for k in MKEYS)
    print(f"  {pair:28s} raw={raw_v:+.3f}   {vals}   -> {LABEL[win]}")

print("\nVariance retained (median, 1.0 = untouched):")
print(f"  {'epigenetic':26s}" +
      "".join(f"  {LABEL[k]}={ratio.loc[ratio.group=='epigenetic', LABEL[k]].median():.3f}" for k in MKEYS))
for m in [x for x in ["ER", "GATA3", "ZEB1", "KI67", "H3S28p"] if x in vr.index]:
    print(f"  {m:26s}" + "".join(f"  {LABEL[k]}={vr.loc[m, LABEL[k]]:.3f}" for k in MKEYS))

print("\nTechnical structure left in the embedding (|rho| tech_ref vs UMAP axes):")
for k in MKEYS:
    r1 = abs(ss.spearmanr(U[k][:, 0], tech_ref).statistic)
    r2 = abs(ss.spearmanr(U[k][:, 1], tech_ref).statistic)
    print(f"  {LABEL[k]:26s} UMAP1={r1:.3f}  UMAP2={r2:.3f}")

In [ ]:
# Rule-based recommendation. Deliberately simple and explicit so it can be audited
# and overridden — it encodes the reasoning, it does not replace judgement.
#
# NB on the floor: empirically EVERY arm lands below it on every cell line tested here,
# so "stayed above the floor" does not discriminate and is deliberately NOT used as a
# filter. It is reported as a degree — how far past the point where biology starts —
# and the choice is made on the retained-variance / induced-correlation trade-off.
epi_var   = {k: ratio.loc[ratio.group == "epigenetic", LABEL[k]].median() for k in MKEYS}
# Signed shift in median pairwise correlation between epigenetic marks, vs raw.
# In practice this is NEGATIVE for every method here (they all dilute the correlation
# structure); a positive value would mean the method manufactured correlation by
# applying one shared factor to every marker. Either direction is distortion, so the
# score penalizes the magnitude — but report the sign, it says which failure it is.
mark_shift = {k: np.median(c_k[k] - c_raw) for k in MKEYS}
frac       = {k: epi_resid[k] / FLOOR for k in MKEYS}

rec_clust = max(MKEYS, key=lambda k: epi_var[k] - abs(mark_shift[k]))

lineage = [m for m in ["ER", "GATA3", "ZEB1"] if m in vr.index]
rec_lin = (max(MKEYS, key=lambda k: vr.loc[lineage, LABEL[k]].mean()) if lineage else None)

print(f"=== {LINE_DISP} — recommendation " + "=" * 38)

above = [k for k in MKEYS if frac[k] >= 0.75]
if above:
    print(f"\nArms stopping at/above the {FLOOR:.3f} biological floor: "
          f"{', '.join(LABEL[k] for k in above)}")
else:
    print(f"\nNo arm stopped at the {FLOOR:.3f} biological floor — all three corrected past the")
    print("point where the untouched surface markers say biology begins. What follows is")
    print("therefore the least-damaging of three over-corrections, not a clean winner.")
print("  " + "  ".join(f"{LABEL[k]}={frac[k]:.2f}x" for k in MKEYS) + "   (1.00x = at the floor)")

print(f"\nFor epigenetic UMAP/clustering : {LABEL[rec_clust]}")
print("  highest retained epigenetic variance minus |distortion of mark-correlation structure|:")
for k in MKEYS:
    mark = "  <-" if k == rec_clust else ""
    print(f"    {LABEL[k]:26s} var={epi_var[k]:.3f}  mark-corr shift={mark_shift[k]:+.3f}  "
          f"score={epi_var[k]-abs(mark_shift[k]):+.3f}{mark}")
print("  (mark-corr shift is signed: negative = correlation structure diluted, which is")
print("   what all three arms do here; nearer zero is better either way.)")

if rec_lin:
    print(f"\nFor {', '.join(lineage)} specifically : {LABEL[rec_lin]}")
    print(f"  mean variance retained {vr.loc[lineage, LABEL[rec_lin]].mean():.3f} vs " +
          ", ".join(f"{vr.loc[lineage, LABEL[k]].mean():.3f} ({LABEL[k]})"
                    for k in MKEYS if k != rec_lin))
    if rec_lin != rec_clust:
        print("  NOTE: differs from the clustering choice. Lineage TFs couple to tech_ref through")
        print("  cell size/lineage rather than permeability, so the arms that correct hardest")
        print("  damage them most. Reading them from a different layer is legitimate.")

worst = min(MKEYS, key=lambda k: frac[k])
print(f"\nMost aggressive arm: {LABEL[worst]} ({frac[worst]:.2f}x floor). Treat its absolute marker")
print("intensities as unreliable — use it only if you specifically want the technical axis")
print("maximally suppressed and accept the biology loss that comes with it.")

**How to act on this.** Layers persist in the project, so you are not forced into one choice: cluster on the
recommended layer and read individual markers from whichever layer preserves them, as long as you say which.
The one thing not to do is quote a marker intensity from an arm flagged as overshooting.

**Caveats.** One cell line, one sample per notebook, so no cross-sample batch effect is tested here — the
methods are being judged on artifact removal within a sample, which is the easier half of the problem. The
surface-marker floor is an indicative benchmark, not a calibrated threshold: surface markers have their own
coupling to cell size, which need not match a histone mark's. This panel has no IdU/CyclinB1/pRb/DNA channel,
so the cell-cycle axis is only observable through `KI67` and `H3S28p`.

## 10. Provenance

In [ ]:
adata = run.read_adata()
nh = uns_history(adata, "normalization")
prov = pd.DataFrame([{
    "method": h["method"], "gamma_mode": h["gamma_mode"],
    "tech_factor_kind": h["tech_factor_kind"], "corrected_layer": h["corrected_layer"],
    "module_version": h["module_version"], "entry_point": h["entry_point"],
    "arcsinh_cofactor": h["arcsinh_cofactor"], "n_cells": h["n_after"],
} for h in nh])
print(f"project : {PROJECT_PATH}")
print(f"run     : {RUN_ID}   ({adata.n_obs:,} cells x {adata.n_vars} markers)")
print(f"layers  : {sorted(adata.layers.keys())}")
print(f"plots   : {PLOTS}")
prov